# Fine-tune a small model on A-S-FLC (Colab + Unsloth)

**Goal:** Teach **Qwen2.5-1.5B** (or similar) to emit valid **DecisionOutput** JSON across all modes (single, security, memory, khmer bilingual).

**Data:** Use `asflc_chat_format.jsonl` from this repo (`training/dataset/`) or load from Hugging Face dataset `denialkhmbot/a-s-flc-decisions` after you run `python training/format_for_hf.py --all` and upload.

**Eval:** Hold out IDs listed in `training/eval_split.json` — do not train on those rows.

## 1. Install dependencies

In [ ]:
%%capture
# Unsloth Colab install (see https://github.com/unslothai/unsloth for latest)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes datasets huggingface_hub

## 2. Login to Hugging Face (for pushing the adapter)

```python
from huggingface_hub import login
login()
```

In [ ]:
# from huggingface_hub import login
# login()

## 3. Load model + tokenizer (4-bit QLoRA)

Adjust `max_seq_length` if you OOM. For DecisionOutput JSON, **2048–4096** is usually enough.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = None  # Auto
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 4. Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 5. Load dataset (chat format)

**Option A — from Hugging Face:**
```python
from datasets import load_dataset
ds = load_dataset("json", data_files="hf://datasets/denialkhmbot/a-s-flc-decisions/asflc_chat_format.jsonl", split="train")
```

**Option B — upload `asflc_chat_format.jsonl` to Colab** and load locally.

In [ ]:
from datasets import load_dataset

# Same 20 IDs as repo training/eval_split.json — never train on these.
EVAL_IDS = frozenset({
    "travel-001", "travel-010", "finance-005", "finance-010", "career-003", "career-010",
    "product-005", "product-010", "health-005", "education-005", "housing-005", "tech-005",
    "relationship-005", "daily-005", "startup-005", "safety-005", "parenting-005",
    "environ-003", "retire-003", "legal-003",
})

USE_HF_REMOTE = True  # False: set LOCAL_JSONL after uploading file to Colab
LOCAL_JSONL = "/content/asflc_chat_format.jsonl"

if USE_HF_REMOTE:
    JSONL_URL = (
        "https://huggingface.co/datasets/denialkhmbot/a-s-flc-decisions/"
        "resolve/main/asflc_chat_format.jsonl"
    )
    ds = load_dataset("json", data_files=JSONL_URL, split="train")
else:
    ds = load_dataset("json", data_files=LOCAL_JSONL, split="train")

eval_ds = ds.filter(lambda ex: ex["id"] in EVAL_IDS)
train_ds = ds.filter(lambda ex: ex["id"] not in EVAL_IDS)
print(f"rows total={len(ds)} train={len(train_ds)} eval_holdout={len(eval_ds)}")

## 6. Map messages → `text`, then train (TRL `SFTTrainer`)

We add a `text` column via `tokenizer.apply_chat_template`, then train with **assistant-only loss masking** (loss is *not* computed on system/user tokens). This uses TRL's `DataCollatorForCompletionOnlyLM` with the Qwen2.5 assistant header token sequence.

Tune **`max_steps`** (e.g. 60 smoke test, 300–800 for stronger fit). If you get `TypeError` about `SFTConfig`, upgrade `trl` or use the fallback block inside the code cell.

In [ ]:
def add_text(batch):
    texts = []
    for msgs in batch["messages"]:
        texts.append(
            tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=False,
            )
        )
    return {"text": texts}

train_tok = train_ds.map(add_text, batched=True, num_proc=2)

from unsloth import is_bfloat16_supported
from trl import SFTTrainer

# Assistant-only loss: only compute loss on assistant tokens, not system/user.
collator = None
try:
    from trl import DataCollatorForCompletionOnlyLM
    response_template = "<|im_start|>assistant\n"
    response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)
    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template_ids,
        tokenizer=tokenizer,
    )
    print("Using assistant-only loss (DataCollatorForCompletionOnlyLM)")
except ImportError:
    print("DataCollatorForCompletionOnlyLM not available, training on full sequence")

try:
    from trl import SFTConfig

    sft_args = SFTConfig(
        output_dir="./outputs_asflc",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=500,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        max_seq_length=max_seq_length,
        dataset_text_field="text",
        packing=False,
    )
    trainer_kwargs = dict(
        model=model,
        args=sft_args,
        train_dataset=train_tok,
        processing_class=tokenizer,
    )
    if collator is not None:
        trainer_kwargs["data_collator"] = collator
    trainer = SFTTrainer(**trainer_kwargs)
except TypeError:
    from transformers import TrainingArguments

    ta = TrainingArguments(
        output_dir="./outputs_asflc",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=500,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    )
    trainer_kwargs = dict(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_tok,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        packing=False,
        args=ta,
    )
    if collator is not None:
        trainer_kwargs["data_collator"] = collator
    trainer = SFTTrainer(**trainer_kwargs)

trainer_stats = trainer.train()

model.save_pretrained("lora_asflc")
tokenizer.save_pretrained("lora_asflc")
print("Saved LoRA to ./lora_asflc")

## 7. After training

The training cell saves **`./lora_asflc`**. Optional: `model.push_to_hub(...)`, merge to 16-bit, or export GGUF — see [Unsloth](https://github.com/unslothai/unsloth).

## 8. Optional — smoke-check one held-out example

Run after training. Parses assistant JSON with `json` only (strict schema check can be done locally with `pydantic`).

In [ ]:
import json

ex = eval_ds[0]
msgs = ex["messages"][:2]
prompt = tokenizer.apply_chat_template(
    msgs,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
out = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=1024,
    use_cache=False,
    do_sample=False,
)
text = tokenizer.decode(out[0], skip_special_tokens=True)
print(text[-1200:])
tail = text.split("assistant")[-1].strip()
try:
    start = tail.index("{")
    json.loads(tail[start:])
    print("\nOK: assistant output contains parseable JSON object")
except Exception as e:
    print("\nParse check:", e)